In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from unidecode import unidecode
import plotly.graph_objects as go
import json
from pathlib import Path
import webbrowser
import joblib

CSV_PATH = "../outputs/prediksi_2024.csv"  # ganti jika nama file CSV berbeda
SHP_PATH = "../maps/BATAS KABUPATEN KOTA DESEMBER 2019 DUKCAPIL.shp"
OUT_HTML = "../outputs/dashboard_peta_perbandingan_prediksi.html"

In [2]:
# ----------------------------------------------
# Normalisasi nama wilayah
# ----------------------------------------------
def norm(s):
    if pd.isna(s): return s
    s = unidecode(str(s)).upper().strip()
    s = s.replace("-", " ").replace("/", " ")
    for w in ["KABUPATEN ", "KOTA ", "KAB. ", "ADM. ", "DAERAH ", "ISTIMEWA "]:
        s = s.replace(w, "")
    return " ".join(s.split())

In [3]:
# ----------------------------------------------
# Load CSV
# ----------------------------------------------
df = pd.read_csv(CSV_PATH)

# Kolom wajib hanya butuh P1 dan Wilayah
req = {"Wilayah", "P1"}
if not req.issubset(df.columns):
    raise ValueError(f"CSV harus punya kolom {req}")

# ----------------------------------------------
# LOAD MODEL LASSO
# ----------------------------------------------
MODEL_PATH = "../models/lasso_model.pkl"

print("[INFO] Loading model LASSO...")
model = joblib.load(MODEL_PATH)
print("[OK] Model LASSO loaded")

# ----------------------------------------------
# Tentukan fitur yang dipakai model
# Sesuaikan dengan fitur training model!
# ----------------------------------------------
feature_cols = ["UHH", "HLS", "RLS", "Pengeluaran", "TPT", "Kepadatan"]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV tidak memiliki fitur yang dibutuhkan model: {missing}")

# ----------------------------------------------
# Generate prediksi dari model
# ----------------------------------------------
print("[INFO] Predicting...")
df["prediksi"] = model.predict(df[feature_cols])
print("[OK] Prediction selesai")

# ----------------------------------------------
# Normalisasi join key
# ----------------------------------------------
df["join_key"] = df["Wilayah"].apply(norm)


[INFO] Loading model LASSO...
[OK] Model LASSO loaded
[INFO] Predicting...
[OK] Prediction selesai


In [4]:
# ----------------------------------------------
# Baca shapefile
# ----------------------------------------------
gdf = gpd.read_file(SHP_PATH)

# Filter ke Jawa Tengah jika shapefile nasional
prov_cols = [c for c in ["PROVINSI","WADMPR","PROPINSI","PROVNAME","NAME_1","provinsi","Provinsi"] if c in gdf.columns]
if prov_cols:
    pc = prov_cols[0]
    gdf["_prov"] = gdf[pc].apply(norm)
    gdf = gdf[gdf["_prov"] == "JAWA TENGAH"].copy()
    if gdf.empty:
        print("[WARN] Tidak ditemukan provinsi 'Jawa Tengah' — mungkin shapefile sudah subset Jateng.")

# Deteksi kolom nama wilayah
candidates = ["NAMOBJ","WADMKC","WADMKK","KABUPATEN","KAB_KOTA","KABKOT","NAMA_KAB","NAME_2","KABKO","WADMKAB"]
name_col = next((c for c in candidates if c in gdf.columns), None)
if name_col is None:
    obj_cols = [c for c in gdf.columns if gdf[c].dtype=='object']
    csv_keys = set(df["join_key"])
    best,score = None,-1
    for c in obj_cols:
        cand = set(gdf[c].astype(str).apply(norm))
        sc = len(cand & csv_keys)
        if sc > score:
            best,score = c,sc
    name_col = best if best else obj_cols[0]

print(f"[INFO] Kolom nama wilayah shapefile: {name_col}")

gdf["join_key"] = gdf[name_col].apply(norm)

# ----------------------------------------------
# Merge shapefile dan data CSV
# ----------------------------------------------
merged = gdf.merge(df[["join_key","P1","prediksi"]], on="join_key", how="left")

print("Match:", merged["P1"].notna().sum(), "/", len(merged))
if merged["P1"].isna().any():
    print("[PERINGATAN] Ada wilayah belum match. Contoh:")
    print(merged.loc[merged["P1"].isna(), [name_col,"join_key"]].drop_duplicates().head(10))


[INFO] Kolom nama wilayah shapefile: KAB_KOTA
Match: 43 / 523
[PERINGATAN] Ada wilayah belum match. Contoh:
          KAB_KOTA         join_key
0             None             None
1       ACEH BARAT       ACEH BARAT
2  ACEH BARAT DAYA  ACEH BARAT DAYA
3       ACEH BESAR       ACEH BESAR
4        ACEH JAYA        ACEH JAYA
5     ACEH SELATAN     ACEH SELATAN
6     ACEH SINGKIL     ACEH SINGKIL
7     ACEH TAMIANG     ACEH TAMIANG
8      ACEH TENGAH      ACEH TENGAH
9    ACEH TENGGARA    ACEH TENGGARA


In [5]:
# ----------------------------------------------
# Pastikan CRS WGS84
# ----------------------------------------------
try:
    if merged.crs is None or merged.crs.to_epsg() != 4326:
        merged = merged.to_crs(4326)
except Exception:
    pass

geojson = json.loads(merged.to_json())


In [6]:
# ----------------------------------------------
# Range nilai
# ----------------------------------------------
vmin = float(min(merged["P1"].min(skipna=True), merged["prediksi"].min(skipna=True)))
vmax = float(max(merged["P1"].max(skipna=True), merged["prediksi"].max(skipna=True)))

mean_p1 = float(merged["P1"].mean(skipna=True))
mean_pred = float(merged["prediksi"].mean(skipna=True))

locations = merged["join_key"]
custom = np.array(merged[name_col].fillna("-"))

hover_p1 = (
    "<b>%{customdata[0]}</b><br>"
    "P1: %{z:.2f} %<br>"
    f"<i>Rata-rata:</i> {mean_p1:.2f} %"
)
hover_pred = (
    "<b>%{customdata[0]}</b><br>"
    "Prediksi: %{z:.2f} %<br>"
    f"<i>Rata-rata:</i> {mean_pred:.2f} %"
)


In [ ]:
# ----------------------------------------------
# Trace P1
# ----------------------------------------------
trace_p1 = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["P1"],
    colorscale="OrRd",
    zmin=vmin, zmax=vmax,
    marker_line_width=0.5,
    marker_line_color="white",
    colorbar_title="P1 (%)",
    hovertemplate=hover_p1,
    customdata=np.c_[custom],
    visible=True
)

# ----------------------------------------------
# Trace Prediksi
# ----------------------------------------------
trace_pred = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["prediksi"],
    colorscale="OrRd",
    zmin=vmin, zmax=vmax,
    marker_line_width=0.5,
    marker_line_color="white",
    colorbar_title="Prediksi (%)",
    hovertemplate=hover_pred,
    customdata=np.c_[custom],
    visible=False
)

title_base = "Indeks Kedalaman Kemiskinan"
subtitle_p1 = f"Provinsi Jawa Tengah • Rata-rata: {mean_p1:.2f}% • Range: {vmin:.2f}–{vmax:.2f}%"
subtitle_pred = f"Provinsi Jawa Tengah • Rata-rata: {mean_pred:.2f}% • Range: {vmin:.2f}–{vmax:.2f}%"

fig = go.Figure(data=[trace_p1, trace_pred])

fig.update_layout(
    title=dict(
        text=f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_p1}</span>",
        x=0.5
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    geo=dict(
        fitbounds="locations",
        visible=False,
        projection_type="mercator"
    ),
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Indeks Kedalaman Kemiskinan (P1)",
                    method="update",
                    args=[
                        {"visible":[True, False]},
                        {"title":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_p1}</span>"}
                    ]
                ),
                dict(
                    label="Prediksi (LASSO)",
                    method="update",
                    args=[
                        {"visible":[False, True]},
                        {"title":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_pred}</span>"}
                    ]
                ),
            ],
            direction="down",
            x=0.05, xanchor="left",
            y=0.95, yanchor="top"
        )
    ],
    annotations=[
        dict(
            x=0.5, y=-0.05, xref="paper", yref="paper",
            text="Sumber: BPS Jawa Tengah (2024) • Prediksi LASSO Model",
            showarrow=False, font=dict(size=11, color="#555")
        )
    ]
)
